<a href="https://colab.research.google.com/github/emmagiometti/Glaciers_GlaciatedLandscapesA26/blob/main/supraglacial_q_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/atedstone/unil_envi_ggl_hydrology_practicals/blob/main/supraglacial_q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical: Interpreting supraglacial melt and discharge

This practical examines three datasets:

1. Ablation at Moulin L41A (Chandler et al., 2021; https://doi.pangaea.de/10.1594/PANGAEA.926842)
2. Meltwater input hydrograph into Moulin L41A (Chandler et al., 2021; https://doi.pangaea.de/10.1594/PANGAEA.926844)
3. Meteorology/surface energy balance at nearby weather station KAN_M (PROMICE; https://doi.org/10.22008/FK2/IW73UU/G9NKYS)

The 'Download' block below downloads both the moulin datasets directly from their original DOI/URL. The weather station data has been prepared specifically for this course, so is downloaded from an UNIL Sharepoint store.

In [ ]:
# Install ipympl so that we can use interactive plots
!pip install ipympl

**Important!** If this is the first time you are running the Notebook, restart your session **now** using the menu 'Runtime... Restart session'. You don't need to run the above cell again afterwards, you can continue with the rest of the Notebook.

In [ ]:
# Activate ipympl in Colab
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# Packages for data download
import requests
import os

In [ ]:
# Packages for data analysis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib widget

## Download the data

In [ ]:
files = {'Moulin_L41A_ice_ablation.tab': 'https://doi.pangaea.de/10.1594/PANGAEA.926842?format=textfile',
         'Moulin_L41A_Q.tab': 'https://doi.pangaea.de/10.1594/PANGAEA.926844?format=textfile',
         'AWS_KAN_M_2012.csv': 'https://unils-my.sharepoint.com/:x:/g/personal/andrew_tedstone_unil_ch/IQD2Lwevmxs2QbywUNCi0sgaAbu2EHvpxQMmLbGxESzF7IM?download=1'
}
for f in files:
    if os.path.exists(f):
        print('File already downloaded.')
    else:
        print('Downloading...')
        response = requests.get(files[f])
        open(f, 'wb').write(response.content)
print('Done.')

## Open the datasets using Pandas

In [ ]:
ablation = pd.read_csv('Moulin_L41A_ice_ablation.tab', sep=r'\t', skiprows=19, parse_dates=True, index_col=0, engine='python')

In [ ]:
discharge = pd.read_csv('Moulin_L41A_Q.tab', sep=r'\t', skiprows=20, parse_dates=True, index_col=0, engine='python')

In [ ]:
aws = pd.read_csv('AWS_KAN_M_2012.csv', parse_dates=True, index_col='time')

## Initial look at dataset structure

In [ ]:
# In Jupyter Notebook, entering the name of a Pandas DataFrame prints its head and foot to screen
ablation

In [ ]:
# We can also get some basic statistics calculated over the whole DataFrame
ablation.describe()

In [ ]:
discharge

Notice that there are three columns of data in the discharge dataset. The columns with 'min' and 'max' in the titles are actually the - and + error bounds corresponding to the 95% confidence intervals. They are *not* the absolute min and max values. See the plotting code below for an example of how to use these columns.

In [ ]:
aws

Here we have measurements acquired by a nearby automatic weather station, the PROMICE KAN_M site. The columns are as follows:

### Surface energy budget variables

All in units of watts per square metre.

* `dlr`: Downwards longwave radiation
* `ulr`: Upwards longwave radiation
* `dsr_cor`: Downwards shortwave radiation, corrected for station tilt etc
* `usr_cor`: Upwards shortwave radiation, corrected for station tilt etc
* `dlhf_u`: Latent heat flux
* `dshf_u`: Sensible heat flux

### Other selected variables

* `snow_height`: as it suggests, the thickness of the snowpack lying on top of the bare ice. Units of metres.
* `z_surf_combined`: total change in surface height, i.e. combining the snow accumulation and melt (which accumulates around the weather station) and the ice melt (which causes the station itself to sink down with the ice surface, therefore ice melt cannot be tracked by the `snow_height` variable). Units of metres.
* `cc`: an estimate of cloud cover, percent.
* `albedo`: in-situ broadband (shortwave) albedo (unitless ratio).

Attention: when looking at `snow_height` and `z_surf_combined`, remember that snow and ice have very different densities - a 1 cm reduction in snow height might only be around 0.5 cm water equivalent, whereas a 1 cm reduction in ice height generates around 0.9 cm water equivalent.

## Plot the data

In [ ]:
# Set up the figure and subplots
plt.figure(figsize=(7,7))
ax_ablat = plt.subplot(411)
ax_seb = plt.subplot(412, sharex=ax_ablat)
ax_alb = plt.subplot(413, sharex=ax_ablat)
ax_disch = plt.subplot(414, sharex=ax_ablat)

# Ablation data
ax_ablat.errorbar(ablation.index+pd.Timedelta(hours=12), ablation['Ablation [mm]'], yerr=ablation['Ablation [±]'],
                  drawstyle='steps-mid', elinewidth=0.7, ecolor='gray')
ax_ablat.grid()
ax_ablat.set_ylabel('Ablation (mm)')

# Surface energy balance data
ax_seb.plot(aws.index, aws.dlhf_u, label='Latent')
ax_seb.plot(aws.index, aws.dshf_u, label='Sensible')
# Calculate net shortwave radiation
swnet = aws.dsr_cor - aws.usr_cor
ax_seb.plot(swnet.index, swnet, label='SWnet')
ax_seb.grid()
ax_seb.set_ylabel('W m-2')

# Albedo
ax_alb.plot(aws.index, aws.albedo, label='Albedo', color='lightgrey')
ax_alb.plot(aws.index, aws.albedo.rolling('24h').mean(), label='Albedo smoothed 24h', linewidth=2)
ax_alb.set_ylim(0, 1)
ax_alb.grid()
ax_alb.set_ylabel('Albedo (0-1)')

# Discharge at the confidence bounds
lower_q = discharge['Q hour mean [m**3/s]'] + discharge['Q hour max [m**3/s]']
upper_q = discharge['Q hour mean [m**3/s]'] - discharge['Q hour min [m**3/s]']

# Plot as a filled area
ax_disch.fill_between(lower_q.index, lower_q, upper_q, alpha=0.5)
discharge['Q hour mean [m**3/s]'].plot(ax=ax_disch)
ax_disch.grid()
ax_disch.set_ylabel(r'Q ($m^3s^{-1}$)')

# To plot vertical lines:
#ax_disch.axvline('2012-07-08', color='red')

# To plot shaded areas:
#ax_disch.axvspan('2012-07-09', '2012-07-11', alpha=0.3, color='red')

## Questions / Exercises

### Describe any relationships you can see between ablation and the main components of the energy budget.

### To what degree does albedo co-vary with the surface energy budget?

### Based only on what you see in these data, do you think that you are looking at snow melt or ice melt?

### How much water flows into the moulin per day, expressed as Olympic swimming pools?

In [ ]:
# Resample the discharge data to daily sums, matching sampling of ablation data
# Note the min_count is set to the number of hours in a day, so that we only
# look at days with full temporal data
# We multiply by 60secs * 60 mins to convert m3s-1 into m3day-1.
q_daily = discharge['Q hour mean [m**3/s]'].resample('1D').sum(min_count=24) * 60 * 60

In [ ]:
vol_pool_m3 = # some value, go find it.....!
# now calculate the mean daily discharge and divide it by this volume.

### Is there a significant correlation between daily ablation and daily discharge?

*To answer this question you will find the code below helpful.*

In [ ]:
import statsmodels.api as sm
# Do an 'Ordinary Least Squares' regression
# See https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html for more information

# Cut ablation data to same time period as discharge (discharge is shorter)
X = ablation.loc[q_daily.index[0]:q_daily.index[-1]]['Ablation [mm]']

# Get the discharge mean series
Y = q_daily

# Some days are missing data; we 'drop' (remove) those days.
model = sm.OLS(Y, sm.add_constant(X), missing='drop')

fit = model.fit()
fit.summary()

Put your answer to this question below. Remember to consider the significance of the test result.

### Is this value what you expected, and why?

### When is peak daily discharge? Why might this be?

To answer this question and the next one, there are at least a couple of approaches that you could use:

- Manual visual interpretation of diurnal values
- Time series resampling -- see the `q_daily` example above, you can supply different functions to the `sum()` operation that we used there.

### When is base (minimum) daily discharge? Why might this be?

### You are following a range of other courses related to data analysis during your studies. What other method(s) might you use to extend this analysis?